In [100]:
import os
from pathlib import Path

import pandas as pd
import polars as pl
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import scipy
import sklearn
import tensorflow as tf
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import mllabs  # ml-labs: our experiment management framework

import sys
print(sys.version)

for i in [pd, pl, np, plt, sns, scipy, sklearn, tf, xgb, lgb, cb, mllabs]:
    if hasattr(i, '__version__'):
        print(i.__name__, i.__version__)
    else:
        print(i.__name__)

3.12.12 (main, Dec 27 2025, 11:08:36) [GCC 13.3.0]
pandas 2.3.3
polars 1.38.1
numpy 2.3.5
matplotlib.pyplot
seaborn 0.13.2
scipy 1.16.3
sklearn 1.8.0
tensorflow 2.20.0
xgboost 3.2.0
lightgbm
catboost 1.2.10
mllabs 0.6.3


In [101]:
from IPython.display import Markdown

from mllabs import Connector, Experimenter, ColSelector
from mllabs.collector import MetricCollector, ModelAttrCollector
from mllabs.adapter import XGBoostAdapter, LightGBMAdapter, CatBoostAdapter
from mllabs.nn import NNClassifier
from mllabs.col import ohe_drop_first

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [102]:
from mllabs.processor import PolarsLoader, PandasConverter, ExprProcessor
from sklearn.pipeline import make_pipeline

data_path = Path('data')

# Binary-encode several categorical features using Polars expressions.
# We also derive indicator columns for internet-service-related features
# ('No internet service' is treated as a separate binary flag).
dict_expr = {
    'gender': (pl.col('gender') == 'Male').cast(pl.Int8),
    'No_Internet': (pl.col('DeviceProtection') == 'No internet service').cast(pl.Int8),
    'DSL_Y': (pl.col('InternetService') == 'DSL').cast(pl.Int8)
}
for i in ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService']:
    dict_expr[i] = (pl.col(i) == 'Yes').cast(pl.Int8)

for i in ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport', 'MultipleLines']:
    dict_expr[i + '_Y'] = (pl.col(i) == 'Yes').cast(pl.Int8)

# PolarsLoader reads CSVs into Polars DataFrames.
# ExprProcessor applies the expression dict above in a single vectorized pass.
loader = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor(dict_expr=dict_expr)
)

df_train = loader.fit_transform([data_path / 'train.csv']).with_columns(
    (pl.col('Churn') == 'Yes').cast(pl.Int8)
)
df_test = loader.transform([data_path / 'test.csv'])

In [104]:
# Variable groupings
# X_bin  : original binary features (Yes/No encoded as 0/1)
# X_tri  : 3-level categorical features (Yes / No / No internet service)
# X_bin2 : derived binary flags from X_tri (e.g. OnlineSecurity_Y)
# X_num  : numerical features
# X_nom  : nominal categoricals kept as strings for tree-based models
X_bin = ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService', 'gender', 'SeniorCitizen']
X_tri = ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport']
X_bin2 = ['{}_Y'.format(i) for i in X_tri] + ['No_Internet', 'DSL_Y', 'MultipleLines_Y']
X_tri.append('InternetService')
X_tri.append('MultipleLines')
X_num = ['TotalCharges', 'MonthlyCharges', 'tenure']
X_nom = ['PaymentMethod', 'Contract']
X_all = X_bin + X_tri + X_bin2 + X_num + X_nom

target = 'Churn'

# Set-up Experimentation

In [105]:
if os.path.exists('exp/ss1'):
    e_model = Experimenter.load('exp/ss1', df_train)
    if e_model.status == 'closed':
        e_model.reopen_exp()
else:
    e_model = Experimenter.create(
        df_train, 'exp/ss1', title='The experimentation for making ML models',
        sp=StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=1),
        sp_v=StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=1),
        splitter_params={'y': target}
    )
Markdown(
    e_model.desc_spec()
)

Loaded: 0 node(s), 0 group(s), 1 fold(s)


## The experimentation for making ML models

| 항목 | 값 |
|------|-----|
| **Outer Splitter (sp)** | `StratifiedShuffleSplit(n_splits=1, random_state=1, test_size=0.2)` |
| **Inner Splitter (sp_v)** | `StratifiedShuffleSplit(n_splits=1, random_state=1, test_size=0.1)` |
| **Splitter Params** | `{y='Churn'}` |
| **Outer Folds** | 1 |
| **Inner Folds** | 1 |

In [6]:
# Pipeline setup
# ml-labs uses a node graph: Stage nodes (preprocessing) → Head nodes (models).
# set_grp defines a group template; set_node creates individual experiment nodes.

e_model.set_grp('pre', role='stage', method='transform')
y_edges = {'y': [(None, target)]}
e_model.set_grp(
    'clf', role='head', method='predict_proba', edges=y_edges
)

# XGBoost — uses XGBoostAdapter to pass eval_set and early_stopping_rounds
e_model.set_grp('xgb', parent='clf', processor=xgb.XGBClassifier,
    adapter=XGBoostAdapter(eval_mode='valid'),
    params={
        'n_estimators': 10000, 'learning_rate': 0.05, 'early_stopping_rounds': 50,
        'eval_metric': 'auc', 'enable_categorical': True, 'device': 'cuda',
        'verbosity': 0, 'random_state': 1,
    }
)
e_model.add_collector(
    ModelAttrCollector('xgb_evals_results', Connector(processor=xgb.XGBClassifier), 'evals_result')
)

# LightGBM — early_stopping passed as dict; LightGBMAdapter creates the callback internally
e_model.set_grp('lgb', parent='clf', processor=lgb.LGBMClassifier,
    adapter=LightGBMAdapter(eval_mode='valid'),
    params={
        'n_estimators': 10000, 'learning_rate': 0.05,
        'early_stopping': {'stopping_rounds': 50, 'first_metric_only': True},
        'eval_metric': 'auc', 'verbose': -1, 'random_state': 1,
    }
)
e_model.add_collector(
    ModelAttrCollector('lgb_evals_results', Connector(processor=lgb.LGBMClassifier), 'evals_result')
)

# CatBoost — CatBoostAdapter handles cat_features and eval_set
e_model.set_grp('cb', parent='clf', processor=cb.CatBoostClassifier,
    adapter=CatBoostAdapter(eval_mode='valid'),
    params={
        'iterations': 10000, 'learning_rate': 0.05, 'early_stopping_rounds': 50,
        'eval_metric': 'AUC', 'verbose': 0, 'random_state': 1, 'task_type': 'GPU',
    }
)
e_model.add_collector(
    ModelAttrCollector('cb_evals_results', Connector('_base$', processor=cb.CatBoostClassifier), 'evals_result')
)

# LogisticRegression
e_model.set_grp('lr', parent='clf', processor=LogisticRegression,
    params={
        'max_iter': 1000, 'random_state': 1,
    }
)

## Neural network
e_model.set_grp('nn', parent = 'clf', processor = NNClassifier, params = {'metrics': ['auc'], 'early_stopping': 10})
e_model.add_collector(
    ModelAttrCollector('nn_evals', Connector(processor=NNClassifier), result_key='evals_result')
)

e_model.add_collector(
    MetricCollector(
        'AUC',
        Connector(edges=y_edges), slice(-1, None), roc_auc_score, include_train = True
    )
)

In [11]:
e_model.set_node('std', processor=StandardScaler, grp='pre', edges={'X': [(None, X_num)]})
e_model.build()

Building 1 node(s)
Build 1/1 (100%)               
Build complete: 1 node(s)


# XGB

In [7]:
e_model.set_grp('xgb_max_depth3', parent='xgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, params = {'max_depth': 3})
for lr in [0.025, 0.05, 0.75]:
    e_model.set_node('xgb_max_depth3_LR{}'.format(lr), grp = 'xgb_max_depth3', params={'learning_rate': lr})
e_model.exp()

Experimenting 3 node(s)
Exp 1/1 (100%)                                                                             
Experimentation complete: 3 node(s)


In [9]:
e_model.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending = False)

,valid,train_sub,valid_sub
xgb_max_depth3_LR0.05,0.915804,0.919165,0.916526
xgb_max_depth3_LR0.025,0.915801,0.919039,0.916529
xgb_max_depth3_LR0.75,0.915064,0.918677,0.915739


In [8]:
e_model.set_grp('xgb_max_depth3_CS', parent='xgb_max_depth3', params = {'learning_rate': 0.05})
for i in [0.25, 0.5, 0.75]:
    e_model.set_node('xgb_max_depth3_CS{}'.format(i), grp = 'xgb_max_depth3_CS', params={'colsample_bytree': i})
e_model.exp()

Experimenting 3 node(s)
Exp 1/1 (100%)                                                                            
Experimentation complete: 3 node(s)


In [11]:
e_model.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending = False).iloc[:5]

,valid,train_sub,valid_sub
xgb_max_depth3_SS0.75,0.915873,0.918819,0.916539
xgb_max_depth3_CS0.75,0.915850,0.919004,0.916492
xgb_max_depth3_LR0.05,0.915796,0.919277,0.916555
xgb_max_depth3_LR0.025,0.915780,0.919035,0.916583
xgb_max_depth3_CS0.5,0.915766,0.918685,0.916340


In [9]:
e_model.set_grp('xgb_max_depth3_SS', parent='xgb_max_depth3_CS', params = {'colsample_bytree': 0.75})
for i in [0.25, 0.5, 0.75]:
    e_model.set_node('xgb_max_depth3_SS{}'.format(i), grp = 'xgb_max_depth3_SS', params={'subsample': i})
e_model.exp()

Experimenting 3 node(s)
Exp 1/1 (100%)                                                                            
Experimentation complete: 3 node(s)


In [12]:
e_model.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending = False).iloc[:5]

,valid,train_sub,valid_sub
xgb_max_depth3_SS0.75,0.915873,0.918819,0.916539
xgb_max_depth3_CS0.75,0.915850,0.919004,0.916492
xgb_max_depth3_LR0.05,0.915796,0.919277,0.916555
xgb_max_depth3_LR0.025,0.915780,0.919035,0.916583
xgb_max_depth3_CS0.5,0.915766,0.918685,0.916340


In [13]:
i = 0.9
e_model.set_node('xgb_max_depth3_SS{}'.format(i), grp = 'xgb_max_depth3_SS', params={'subsample': i})
e_model.exp()

Experimenting 1 node(s)
Exp 1/1 (100%)                                                                          
Experimentation complete: 1 node(s)


In [15]:
e_model.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending = False).iloc[:5]

,valid,train_sub,valid_sub
xgb_max_depth3_SS0.9,0.915921,0.919100,0.916536
xgb_max_depth3_CS0.75,0.915902,0.919230,0.916541
xgb_max_depth3_SS0.75,0.915875,0.918805,0.916555
xgb_max_depth3_LR0.05,0.915804,0.919165,0.916526
xgb_max_depth3_LR0.025,0.915801,0.919039,0.916529


In [14]:
e_model.set_grp('xgb_max_depth3_l2', parent='xgb_max_depth3_SS', params = {'subsample': 0.9})
for i in [0.01, 0.1, 10]:
    e_model.set_node('xgb_max_depth3_l2{}'.format(i), grp = 'xgb_max_depth3_l2', params={'reg_lambda': i})
e_model.exp()

Experimenting 3 node(s)
Exp 1/1 (100%)                                                                            
Experimentation complete: 3 node(s)


In [15]:
e_model.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending = False).iloc[:5]

,valid,train_sub,valid_sub
xgb_max_depth3_l20.01,0.915938,0.919178,0.916546
xgb_max_depth3_SS0.9,0.915913,0.919105,0.916534
xgb_max_depth3_l20.1,0.915904,0.919105,0.916577
xgb_max_depth3_l210,0.915883,0.918822,0.916492
xgb_max_depth3_SS0.75,0.915873,0.918819,0.916539


In [16]:
e_model.set_grp('xgb_max_depth3_l1', parent='xgb_max_depth3_SS', params = {'reg_lambda': 10})
for i in [0.01, 0.1, 10]:
    e_model.set_node('xgb_max_depth3_l1{}'.format(i), grp = 'xgb_max_depth3_l1', params={'reg_alpha': i})
e_model.exp()

Experimenting 3 node(s)
Exp 1/1 (100%)                                                                           
Experimentation complete: 3 node(s)


In [18]:
e_model.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending = False).iloc[:5]

,valid,train_sub,valid_sub
xgb_max_depth3_l20.01,0.915938,0.919178,0.916546
xgb_max_depth3_SS0.9,0.915913,0.919105,0.916534
xgb_max_depth3_l20.1,0.915904,0.919105,0.916577
xgb_max_depth3_l210,0.915883,0.918822,0.916492
xgb_max_depth3_l10.1,0.915878,0.918944,0.916498


In [19]:
e_model.set_grp('xgb_max_depth4', parent='xgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
                params = {'max_depth': 4, 'colsample_bytree': 0.75, 'subsample': 0.9, 'reg_lambda': 10})
for lr in [0.025, 0.05, 0.75]:
    e_model.set_node('xgb_max_depth4_LR{}'.format(lr), grp = 'xgb_max_depth4', params={'learning_rate': lr})
e_model.exp()

Experimenting 3 node(s)
Exp 1/1 (100%)                                                                             
Experimentation complete: 3 node(s)


In [20]:
e_model.get_collector('AUC').get_metrics_agg("xgb_max_depth4")[0].sort_values('valid', ascending = False).iloc[:5]

,valid,train_sub,valid_sub
xgb_max_depth4_LR0.05,0.915902,0.919906,0.916591
xgb_max_depth4_LR0.025,0.915842,0.919517,0.916569
xgb_max_depth4_LR0.75,0.914612,0.917661,0.915636


In [21]:
e_model.set_grp('xgb_max_depth5', parent='xgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
                params = {'max_depth': 5, 'colsample_bytree': 0.75, 'subsample': 0.9, 'reg_lambda': 10})
for lr in [0.025, 0.05, 0.75]:
    e_model.set_node('xgb_max_depth5_LR{}'.format(lr), grp = 'xgb_max_depth5', params={'learning_rate': lr})
e_model.exp()

Experimenting 3 node(s)
Exp 1/1 (100%)                                                                             
Experimentation complete: 3 node(s)


In [22]:
e_model.get_collector('AUC').get_metrics_agg("xgb_max_depth5")[0].sort_values('valid', ascending = False).iloc[:5]

,valid,train_sub,valid_sub
xgb_max_depth5_LR0.025,0.915872,0.921263,0.916623
xgb_max_depth5_LR0.05,0.915836,0.921066,0.916504
xgb_max_depth5_LR0.75,0.914377,0.919628,0.915147


In [23]:
i = 2
e_model.set_node('xgb_max_depth{}'.format(i), grp = 'xgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
                     params={'max_depth': i, 'learning_rate': 0.05, 'colsample_bytree': 0.75, 'subsample': 0.9, 'reg_lambda': 10})
for i in range(6, 10):
    e_model.set_node('xgb_max_depth{}'.format(i), grp = 'xgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
                     params={'max_depth': i, 'learning_rate': 0.025, 'colsample_bytree': 0.75, 'subsample': 0.9, 'reg_lambda': 10})
e_model.exp()

Experimenting 5 node(s)
Exp 1/1 (100%)                                                                     
Experimentation complete: 5 node(s)


In [24]:
e_model.get_collector('AUC').get_metrics_agg("xgb_max_depth2.*")[0]

,valid,train_sub,valid_sub
xgb_max_depth2,0.915647,0.917677,0.916265


In [25]:
e_model.get_collector('AUC').get_metrics_agg("xgb_.*")[0].sort_values('valid', ascending = False).iloc[:5]

,valid,train_sub,valid_sub
xgb_max_depth3_l20.01,0.915938,0.919178,0.916546
xgb_max_depth3_SS0.9,0.915913,0.919105,0.916534
xgb_max_depth3_l20.1,0.915904,0.919105,0.916577
xgb_max_depth4_LR0.05,0.915902,0.919906,0.916591
xgb_max_depth3_l210,0.915883,0.918822,0.916492


In [26]:
d = e_model.collectors['xgb_evals_results'].get_attrs()
s_best_iteration = pd.Series({
    k: np.mean(
        [[j.unstack().unstack()[('validation_0', 'auc')].argmax() for j in i] for i in v]
    )
    for k, v in d.items()
}, name=('evals_result', 'best_iteration'))
e_model.get_collector('AUC').get_metrics_agg("xgb_.*")[0].sort_values('valid', ascending = False).join(
    s_best_iteration.rename('best_iteration')
)

,valid,train_sub,valid_sub,best_iteration
xgb_max_depth3_l20.01,0.915938,0.919178,0.916546,2312.0
xgb_max_depth3_SS0.9,0.915913,0.919105,0.916534,2312.0
xgb_max_depth3_l20.1,0.915904,0.919105,0.916577,2233.0
xgb_max_depth4_LR0.05,0.915902,0.919906,0.916591,1428.0
xgb_max_depth3_l210,0.915883,0.918822,0.916492,2313.0
xgb_max_depth3_l10.1,0.915878,0.918944,0.916498,2658.0
xgb_max_depth3_SS0.75,0.915873,0.918819,0.916539,2031.0
xgb_max_depth5_LR0.025,0.915872,0.921263,0.916623,2036.0
xgb_max_depth3_CS0.75,0.915850,0.919004,0.916492,2375.0
xgb_max_depth3_l10.01,0.915849,0.918881,0.916487,2576.0


In [27]:
e_model.finalize(None)

Finalize 'xgb_max_depth3_LR0.025'
Finalize 'xgb_max_depth3_LR0.05'
Finalize 'xgb_max_depth3_LR0.75'
Finalize 'xgb_max_depth3_CS0.25'
Finalize 'xgb_max_depth3_CS0.5'
Finalize 'xgb_max_depth3_CS0.75'
Finalize 'xgb_max_depth3_SS0.25'
Finalize 'xgb_max_depth3_SS0.5'
Finalize 'xgb_max_depth3_SS0.75'
Finalize 'xgb_max_depth3_SS0.9'
Finalize 'xgb_max_depth3_l20.01'
Finalize 'xgb_max_depth3_l20.1'
Finalize 'xgb_max_depth3_l210'
Finalize 'xgb_max_depth3_l10.01'
Finalize 'xgb_max_depth3_l10.1'
Finalize 'xgb_max_depth3_l110'
Finalize 'xgb_max_depth4_LR0.025'
Finalize 'xgb_max_depth4_LR0.05'
Finalize 'xgb_max_depth4_LR0.75'
Finalize 'xgb_max_depth5_LR0.025'
Finalize 'xgb_max_depth5_LR0.05'
Finalize 'xgb_max_depth5_LR0.75'
Finalize 'xgb_max_depth2'
Finalize 'xgb_max_depth6'
Finalize 'xgb_max_depth7'
Finalize 'xgb_max_depth8'
Finalize 'xgb_max_depth9'


# LGB

In [6]:
e_model.set_grp('lgb_nl_7', parent='lgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, params = {'num_leaves': 7})
for lr in [0.025, 0.05, 0.75]:
    e_model.set_node('lgb_nl_LR{}'.format(lr), grp = 'lgb_nl_7', params={'learning_rate': lr})
e_model.exp()

Experimenting 3 node(s)
Exp 0/1 (0%) > lgb_nl_LR0.025 0/3 (0%) > 1/10000 (0%) valid_0-auc: 0.8761, valid_0-binary_logloss: 0.5244Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_LR0.025 0/3 (0%) > 4000/10000 (40%) valid_0-auc: 0.9163, valid_0-binary_logloss: 0.2979Early stopping, best iteration is:
[4248]	valid_0's auc: 0.916345	valid_0's binary_logloss: 0.297871
Evaluated only: auc
Exp 0/1 (0%) > lgb_nl_LR0.05 1/3 (33%) > 1/10000 (0%) valid_0-auc: 0.8761, valid_0-binary_logloss: 0.5156Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_LR0.05 1/3 (33%) > 1000/10000 (10%) valid_0-auc: 0.9157, valid_0-binary_logloss: 0.2988Early stopping, best iteration is:
[1812]	valid_0's auc: 0.916266	valid_0's binary_logloss: 0.297969
Evaluated only: auc
Exp 0/1 (0%) > lgb_nl_LR0.75 2/3 (66%) > 1/10000 (0%) valid_0-auc: 0.8761, valid_0-binary_logloss: 0.3737Training until validation scores don't improve for 50 rounds
Early stopping, be

In [7]:
e_model.get_collector('AUC').get_metrics_agg("lgb_.*")[0].sort_values('valid', ascending = False).iloc[:5]

,valid,train_sub,valid_sub
lgb_nl_LR0.025,0.915456,0.919561,0.916345
lgb_nl_LR0.05,0.915420,0.919132,0.916266
lgb_nl_LR0.75,0.913622,0.917432,0.914866


In [8]:
lr = 0.01
e_model.set_node('lgb_nl_LR{}'.format(lr), grp = 'lgb_nl_7', params={'learning_rate': lr})
e_model.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb_nl_LR0.01 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.8761, valid_0-binary_logloss: 0.5298Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_LR0.01 0/1 (0%) > 10000/10000 (100%) valid_0-auc: 0.9164, valid_0-binary_logloss: 0.2978Did not meet early stopping. Best iteration is:
[9997]	valid_0's auc: 0.916369	valid_0's binary_logloss: 0.297827
Evaluated only: auc
Exp 1/1 (100%)                                                                                                
Experimentation complete: 1 node(s)


In [9]:
e_model.get_collector('AUC').get_metrics_agg("lgb_.*")[0].sort_values('valid', ascending = False).iloc[:5]

,valid,train_sub,valid_sub
lgb_nl_LR0.025,0.915456,0.919561,0.916345
lgb_nl_LR0.05,0.915420,0.919132,0.916266
lgb_nl_LR0.01,0.915413,0.919343,0.916369
lgb_nl_LR0.75,0.913622,0.917432,0.914866


In [10]:
e_model.set_grp('lgb_nl_31', parent='lgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, params = {'num_leaves': 31})
for lr in [0.01, 0.025, 0.05, 0.75]:
    e_model.set_node('lgb_nl_31_LR{}'.format(lr), grp = 'lgb_nl_31', params={'learning_rate': lr})
e_model.exp()

Experimenting 4 node(s)
Exp 0/1 (0%) > lgb_nl_31_LR0.025 0/4 (0%) > 1/10000 (0%) valid_0-auc: 0.9016, valid_0-binary_logloss: 0.5232Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_31_LR0.025 0/4 (0%) > 1000/10000 (10%) valid_0-auc: 0.9161, valid_0-binary_logloss: 0.2983Early stopping, best iteration is:
[1344]	valid_0's auc: 0.916188	valid_0's binary_logloss: 0.298138
Evaluated only: auc
Exp 0/1 (0%) > lgb_nl_31_LR0.01 1/4 (25%) > 1/10000 (0%) valid_0-auc: 0.9016, valid_0-binary_logloss: 0.5293Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_31_LR0.01 1/4 (25%) > 3000/10000 (30%) valid_0-auc: 0.9161, valid_0-binary_logloss: 0.2982Early stopping, best iteration is:
[3849]	valid_0's auc: 0.916245	valid_0's binary_logloss: 0.298042
Evaluated only: auc
Exp 0/1 (0%) > lgb_nl_31_LR0.05 2/4 (50%) > 1/10000 (0%) valid_0-auc: 0.9016, valid_0-binary_logloss: 0.5132Training until validation scores don't improve for 50 rounds
Ear

In [11]:
e_model.get_collector('AUC').get_metrics_agg("lgb_.*")[0].sort_values('valid', ascending = False).iloc[:5]

,valid,train_sub,valid_sub
lgb_nl_LR0.025,0.915456,0.919561,0.916345
lgb_nl_LR0.05,0.915420,0.919132,0.916266
lgb_nl_LR0.01,0.915413,0.919343,0.916369
lgb_nl_31_LR0.01,0.915181,0.923464,0.916245
lgb_nl_31_LR0.025,0.915152,0.922711,0.916188


In [12]:
e_model.set_grp('lgb_nl_15', parent='lgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, params = {'num_leaves': 15})
for lr in [0.025, 0.05, 0.75]:
    e_model.set_node('lgb_nl_15_LR{}'.format(lr), grp = 'lgb_nl_15', params={'learning_rate': lr})
e_model.exp()

Experimenting 3 node(s)
Exp 0/1 (0%) > lgb_nl_15_LR0.025 0/3 (0%) > 1/10000 (0%) valid_0-auc: 0.8927, valid_0-binary_logloss: 0.5235Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_15_LR0.025 0/3 (0%) > 2000/10000 (20%) valid_0-auc: 0.9162, valid_0-binary_logloss: 0.2982Early stopping, best iteration is:
[2477]	valid_0's auc: 0.91624	valid_0's binary_logloss: 0.298052
Evaluated only: auc
Exp 0/1 (0%) > lgb_nl_15_LR0.75 1/3 (33%) > 1/10000 (0%) valid_0-auc: 0.8927, valid_0-binary_logloss: 0.3584Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[51]	valid_0's auc: 0.914642	valid_0's binary_logloss: 0.301843
Evaluated only: auc
Exp 0/1 (0%) > lgb_nl_15_LR0.05 2/3 (66%) > 1/10000 (0%) valid_0-auc: 0.8927, valid_0-binary_logloss: 0.5138Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_15_LR0.05 2/3 (66%) > 1000/10000 (10%) valid_0-auc: 0.9161, valid_0-binary_logloss: 0.2982Early 

In [13]:
e_model.get_collector('AUC').get_metrics_agg("lgb_.*")[0].sort_values('valid', ascending = False).iloc[:5]

,valid,train_sub,valid_sub
lgb_nl_LR0.025,0.915456,0.919561,0.916345
lgb_nl_LR0.05,0.915420,0.919132,0.916266
lgb_nl_LR0.01,0.915413,0.919343,0.916369
lgb_nl_15_LR0.05,0.915251,0.921375,0.916201
lgb_nl_31_LR0.01,0.915181,0.923464,0.916245


In [14]:
e_model.set_node('lgb_nl_15_msf64'.format(lr), grp = 'lgb_nl_15', params={'learning_rate': 0.05, 'min_samples_leaf': 64})
e_model.set_node('lgb_nl_15_msf128'.format(lr), grp = 'lgb_nl_15', params={'learning_rate': 0.05, 'min_samples_leaf': 128})
e_model.set_node('lgb_nl_15_msf256'.format(lr), grp = 'lgb_nl_15', params={'learning_rate': 0.05, 'min_samples_leaf': 256})
e_model.exp()

Experimenting 3 node(s)
Exp 0/1 (0%) > lgb_nl_15_msf128 0/3 (0%) > 1/10000 (0%) valid_0-auc: 0.8927, valid_0-binary_logloss: 0.5138Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_15_msf128 0/3 (0%) > 1000/10000 (10%) valid_0-auc: 0.9163, valid_0-binary_logloss: 0.2980Early stopping, best iteration is:
[1019]	valid_0's auc: 0.916263	valid_0's binary_logloss: 0.298034
Evaluated only: auc
Exp 0/1 (0%) > lgb_nl_15_msf256 1/3 (33%) > 1/10000 (0%) valid_0-auc: 0.8927, valid_0-binary_logloss: 0.5138Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_15_msf256 1/3 (33%) > 1000/10000 (10%) valid_0-auc: 0.9162, valid_0-binary_logloss: 0.2981Early stopping, best iteration is:
[963]	valid_0's auc: 0.91623	valid_0's binary_logloss: 0.29809
Evaluated only: auc
Exp 0/1 (0%) > lgb_nl_15_msf64 2/3 (66%) > 1/10000 (0%) valid_0-auc: 0.8927, valid_0-binary_logloss: 0.5138Training until validation scores don't improve for 50 rounds
Exp 0/1 (

In [15]:
e_model.get_collector('AUC').get_metrics_agg("lgb_nl_15_msf.*")[0].sort_values('valid', ascending = False)

,valid,train_sub,valid_sub
lgb_nl_15_msf256,0.915358,0.919778,0.916230
lgb_nl_15_msf128,0.915319,0.920199,0.916263
lgb_nl_15_msf64,0.915255,0.920729,0.916288


In [16]:
e_model.set_grp('lgb_nl_3', parent='lgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, params = {'num_leaves': 3})
for lr in [0.025, 0.05, 0.75]:
    e_model.set_node('lgb_nl_3_LR{}'.format(lr), grp = 'lgb_nl_3', params={'learning_rate': lr})
e_model.exp()

Experimenting 3 node(s)
Exp 0/1 (0%) > lgb_nl_3_LR0.75 0/3 (0%) > 1/10000 (0%) valid_0-auc: 0.8052, valid_0-binary_logloss: 0.4120Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[503]	valid_0's auc: 0.916151	valid_0's binary_logloss: 0.298235
Evaluated only: auc
Exp 0/1 (0%) > lgb_nl_3_LR0.025 1/3 (33%) > 1/10000 (0%) valid_0-auc: 0.8052, valid_0-binary_logloss: 0.5265Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_3_LR0.025 1/3 (33%) > 10000/10000 (100%) valid_0-auc: 0.9160, valid_0-binary_logloss: 0.2985Did not meet early stopping. Best iteration is:
[9997]	valid_0's auc: 0.91597	valid_0's binary_logloss: 0.298483
Evaluated only: auc
Exp 0/1 (0%) > lgb_nl_3_LR0.05 2/3 (66%) > 1/10000 (0%) valid_0-auc: 0.8052, valid_0-binary_logloss: 0.5197Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_3_LR0.05 2/3 (66%) > 4000/10000 (40%) valid_0-auc: 0.9158, valid_0-binary_logloss: 

In [17]:
e_model.get_collector('AUC').get_metrics_agg("lgb_nl_3_LR.*")[0].sort_values('valid', ascending = False)

,valid,train_sub,valid_sub
lgb_nl_3_LR0.75,0.915304,0.917651,0.916151
lgb_nl_3_LR0.025,0.915226,0.917142,0.915970
lgb_nl_3_LR0.05,0.915144,0.916986,0.915863


In [18]:
e_model.set_node('lgb_nl_7_msf256'.format(lr), grp = 'lgb_nl_7', params={'learning_rate': 0.05, 'min_samples_leaf': 256})
e_model.set_node('lgb_nl_7_msf512'.format(lr), grp = 'lgb_nl_7', params={'learning_rate': 0.05, 'min_samples_leaf': 512})
e_model.exp()

Experimenting 2 node(s)
Exp 0/1 (0%) > lgb_nl_7_msf256 0/2 (0%) > 1/10000 (0%) valid_0-auc: 0.8761, valid_0-binary_logloss: 0.5156Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_7_msf256 0/2 (0%) > 1000/10000 (10%) valid_0-auc: 0.9158, valid_0-binary_logloss: 0.2988Early stopping, best iteration is:
[1738]	valid_0's auc: 0.916235	valid_0's binary_logloss: 0.29804
Evaluated only: auc
Exp 0/1 (0%) > lgb_nl_7_msf512 1/2 (50%) > 1/10000 (0%) valid_0-auc: 0.8761, valid_0-binary_logloss: 0.5156Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_7_msf512 1/2 (50%) > 2000/10000 (20%) valid_0-auc: 0.9164, valid_0-binary_logloss: 0.2979Early stopping, best iteration is:
[2072]	valid_0's auc: 0.916379	valid_0's binary_logloss: 0.29783
Evaluated only: auc
Exp 1/1 (100%)                                                                                                 
Experimentation complete: 2 node(s)


In [19]:
e_model.get_collector('AUC').get_metrics_agg("lgb_.*")[0].sort_values('valid', ascending = False).iloc[:5]

,valid,train_sub,valid_sub
lgb_nl_7_msf512,0.915573,0.918925,0.916379
lgb_nl_LR0.025,0.915456,0.919561,0.916345
lgb_nl_7_msf256,0.915438,0.918634,0.916235
lgb_nl_LR0.05,0.915420,0.919132,0.916266
lgb_nl_LR0.01,0.915413,0.919343,0.916369


In [20]:
e_model.set_node('lgb_nl_7_msf512_cs0.5'.format(lr), grp = 'lgb_nl_7', 
                 params={'learning_rate': 0.05, 'min_samples_leaf': 512, 'colsample_bytree': 0.5})
e_model.set_node('lgb_nl_7_msf512_cs0.75'.format(lr), grp = 'lgb_nl_7', 
                 params={'learning_rate': 0.05, 'min_samples_leaf': 512, 'colsample_bytree': 0.75})
e_model.exp()

Experimenting 2 node(s)
Exp 0/1 (0%) > lgb_nl_7_msf512_cs0.5 0/2 (0%) > 1/10000 (0%) valid_0-auc: 0.8702, valid_0-binary_logloss: 0.5165Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_7_msf512_cs0.5 0/2 (0%) > 2000/10000 (20%) valid_0-auc: 0.9164, valid_0-binary_logloss: 0.2978Early stopping, best iteration is:
[2426]	valid_0's auc: 0.916531	valid_0's binary_logloss: 0.297559
Evaluated only: auc
Exp 0/1 (0%) > lgb_nl_7_msf512_cs0.75 1/2 (50%) > 1/10000 (0%) valid_0-auc: 0.8724, valid_0-binary_logloss: 0.5164Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_7_msf512_cs0.75 1/2 (50%) > 2000/10000 (20%) valid_0-auc: 0.9163, valid_0-binary_logloss: 0.2979Early stopping, best iteration is:
[2370]	valid_0's auc: 0.916374	valid_0's binary_logloss: 0.297817
Evaluated only: auc
Exp 1/1 (100%)                                                                                                        
Experimentation complete: 2 node(

In [21]:
e_model.get_collector('AUC').get_metrics_agg("lgb_.*")[0].sort_values('valid', ascending = False).iloc[:5]

,valid,train_sub,valid_sub
lgb_nl_7_msf512_cs0.75,0.915677,0.919118,0.916374
lgb_nl_7_msf512_cs0.5,0.915643,0.918912,0.916531
lgb_nl_7_msf512,0.915573,0.918925,0.916379
lgb_nl_LR0.025,0.915456,0.919561,0.916345
lgb_nl_7_msf256,0.915438,0.918634,0.916235


In [22]:
e_model.set_node('lgb_nl_7_msf512_cs0.75_ss0.5'.format(lr), grp = 'lgb_nl_7', 
                 params={'learning_rate': 0.05, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 'subsample': 0.5, 'subsample_freq': 1})
e_model.set_node('lgb_nl_7_msf512_cs0.75_ss0.75'.format(lr), grp = 'lgb_nl_7', 
                 params={'learning_rate': 0.05, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 'subsample': 0.75, 'subsample_freq': 1})
e_model.set_node('lgb_nl_7_msf512_cs0.75_ss0.9'.format(lr), grp = 'lgb_nl_7', 
                 params={'learning_rate': 0.05, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 'subsample': 0.9, 'subsample_freq': 1})
e_model.exp()

Experimenting 3 node(s)
Exp 0/1 (0%) > lgb_nl_7_msf512_cs0.75_ss0.75 0/3 (0%) > 1/10000 (0%) valid_0-auc: 0.8724, valid_0-binary_logloss: 0.5164Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_7_msf512_cs0.75_ss0.75 0/3 (0%) > 2000/10000 (20%) valid_0-auc: 0.9165, valid_0-binary_logloss: 0.2977Early stopping, best iteration is:
[2434]	valid_0's auc: 0.916561	valid_0's binary_logloss: 0.297532
Evaluated only: auc
Exp 0/1 (0%) > lgb_nl_7_msf512_cs0.75_ss0.5 1/3 (33%) > 1/10000 (0%) valid_0-auc: 0.8724, valid_0-binary_logloss: 0.5164Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_7_msf512_cs0.75_ss0.5 1/3 (33%) > 1000/10000 (10%) valid_0-auc: 0.9158, valid_0-binary_logloss: 0.2987Early stopping, best iteration is:
[1652]	valid_0's auc: 0.916267	valid_0's binary_logloss: 0.298022
Evaluated only: auc
Exp 0/1 (0%) > lgb_nl_7_msf512_cs0.75_ss0.9 2/3 (66%) > 1/10000 (0%) valid_0-auc: 0.8724, valid_0-binary_logloss: 0.5164Trai

In [23]:
e_model.get_collector('AUC').get_metrics_agg("lgb_.*")[0].sort_values('valid', ascending = False).iloc[:5]

,valid,train_sub,valid_sub
lgb_nl_7_msf512_cs0.75_ss0.75,0.915727,0.919305,0.916561
lgb_nl_7_msf512_cs0.75_ss0.9,0.915693,0.919275,0.916431
lgb_nl_7_msf512_cs0.75,0.915677,0.919118,0.916374
lgb_nl_7_msf512_cs0.5,0.915643,0.918912,0.916531
lgb_nl_7_msf512,0.915573,0.918925,0.916379


In [24]:
e_model.set_node('lgb_nl_7_msf512_cs0.75_ss0.75_l21'.format(lr), grp = 'lgb_nl_7', 
                 params={'learning_rate': 0.05, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 'subsample': 0.75, 'subsample_freq': 1,
                        'reg_lambda': 1})
e_model.set_node('lgb_nl_7_msf512_cs0.75_ss0.75_l210'.format(lr), grp = 'lgb_nl_7', 
                 params={'learning_rate': 0.05, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 'subsample': 0.75, 'subsample_freq': 1,
                        'reg_lambda': 10})
e_model.exp()

Experimenting 2 node(s)
Exp 0/1 (0%) > lgb_nl_7_msf512_cs0.75_ss0.75_l210 0/2 (0%) > 1/10000 (0%) valid_0-auc: 0.8724, valid_0-binary_logloss: 0.5165Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_7_msf512_cs0.75_ss0.75_l210 0/2 (0%) > 2000/10000 (20%) valid_0-auc: 0.9165, valid_0-binary_logloss: 0.2977Early stopping, best iteration is:
[2481]	valid_0's auc: 0.91657	valid_0's binary_logloss: 0.297511
Evaluated only: auc
Exp 0/1 (0%) > lgb_nl_7_msf512_cs0.75_ss0.75_l21 1/2 (50%) > 1/10000 (0%) valid_0-auc: 0.8724, valid_0-binary_logloss: 0.5164Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_7_msf512_cs0.75_ss0.75_l21 1/2 (50%) > 2000/10000 (20%) valid_0-auc: 0.9164, valid_0-binary_logloss: 0.2977Early stopping, best iteration is:
[2457]	valid_0's auc: 0.916529	valid_0's binary_logloss: 0.297591
Evaluated only: auc
Exp 1/1 (100%)                                                                                           

In [25]:
e_model.set_node('lgb_nl_15_msf512_cs0.75_ss0.75_l21'.format(lr), grp = 'lgb_nl_15', 
                 params={'learning_rate': 0.05, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 'subsample': 0.75, 'subsample_freq': 1,
                        'reg_lambda': 1})
e_model.set_node('lgb_nl_15_msf512_cs0.75_ss0.75_l21_lr0.025'.format(lr), grp = 'lgb_nl_15', 
                 params={'learning_rate': 0.025, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 'subsample': 0.75, 'subsample_freq': 1,
                        'reg_lambda': 1})
e_model.exp()

Experimenting 2 node(s)
Exp 0/1 (0%) > lgb_nl_15_msf512_cs0.75_ss0.75_l21_lr0.025 0/2 (0%) > 1/10000 (0%) valid_0-auc: 0.8840, valid_0-binary_logloss: 0.5246Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_15_msf512_cs0.75_ss0.75_l21_lr0.025 0/2 (0%) > 2000/10000 (20%) valid_0-auc: 0.9165, valid_0-binary_logloss: 0.2976Early stopping, best iteration is:
[1994]	valid_0's auc: 0.916526	valid_0's binary_logloss: 0.297586
Evaluated only: auc
Exp 0/1 (0%) > lgb_nl_15_msf512_cs0.75_ss0.75_l21 1/2 (50%) > 1/10000 (0%) valid_0-auc: 0.8840, valid_0-binary_logloss: 0.5159Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_nl_15_msf512_cs0.75_ss0.75_l21 1/2 (50%) > 1000/10000 (10%) valid_0-auc: 0.9164, valid_0-binary_logloss: 0.2978Early stopping, best iteration is:
[1058]	valid_0's auc: 0.916436	valid_0's binary_logloss: 0.297711
Evaluated only: auc
Exp 1/1 (100%)                                                                        

In [26]:
e_model.get_collector('AUC').get_metrics_agg("lgb_.*")[0].sort_values('valid', ascending = False).iloc[:5]

,valid,train_sub,valid_sub
lgb_nl_7_msf512_cs0.75_ss0.75_l21,0.915752,0.919291,0.916529
lgb_nl_7_msf512_cs0.75_ss0.75_l210,0.915750,0.919063,0.916570
lgb_nl_7_msf512_cs0.75_ss0.75,0.915727,0.919305,0.916561
lgb_nl_15_msf512_cs0.75_ss0.75_l21_lr0.025,0.915707,0.919579,0.916526
lgb_nl_7_msf512_cs0.75_ss0.9,0.915693,0.919275,0.916431


In [27]:
d = e_model.collectors['lgb_evals_results'].get_attrs()
s_best_iteration = pd.Series({
    k: np.mean(
        [[j.unstack().unstack()[('valid_0', 'auc')].argmax() for j in i] for i in v]
    )
    for k, v in d.items()
}, name=('evals_result', 'best_iteration'))
e_model.get_collector('AUC').get_metrics_agg("lgb_.*")[0].sort_values('valid', ascending = False).join(
    s_best_iteration.rename('best_iteration')
)

,valid,train_sub,valid_sub,best_iteration
lgb_nl_7_msf512_cs0.75_ss0.75_l21,0.915752,0.919291,0.916529,2456.0
lgb_nl_7_msf512_cs0.75_ss0.75_l210,0.915750,0.919063,0.916570,2480.0
lgb_nl_7_msf512_cs0.75_ss0.75,0.915727,0.919305,0.916561,2433.0
lgb_nl_15_msf512_cs0.75_ss0.75_l21_lr0.025,0.915707,0.919579,0.916526,1993.0
lgb_nl_7_msf512_cs0.75_ss0.9,0.915693,0.919275,0.916431,2368.0
lgb_nl_7_msf512_cs0.75,0.915677,0.919118,0.916374,2369.0
lgb_nl_15_msf512_cs0.75_ss0.75_l21,0.915673,0.919695,0.916436,1057.0
lgb_nl_7_msf512_cs0.5,0.915643,0.918912,0.916531,2425.0
lgb_nl_7_msf512,0.915573,0.918925,0.916379,2071.0
lgb_nl_7_msf512_cs0.75_ss0.5,0.915515,0.918344,0.916267,1651.0


In [12]:
e_model.finalize('lgb_.*')

Finalize 'lgb_nl_LR0.025'
Finalize 'lgb_nl_LR0.05'
Finalize 'lgb_nl_LR0.75'
Finalize 'lgb_nl_LR0.01'
Finalize 'lgb_nl_31_LR0.01'
Finalize 'lgb_nl_31_LR0.025'
Finalize 'lgb_nl_31_LR0.05'
Finalize 'lgb_nl_31_LR0.75'
Finalize 'lgb_nl_15_LR0.025'
Finalize 'lgb_nl_15_LR0.05'
Finalize 'lgb_nl_15_LR0.75'
Finalize 'lgb_nl_15_msf64'
Finalize 'lgb_nl_15_msf128'
Finalize 'lgb_nl_15_msf256'
Finalize 'lgb_nl_3_LR0.025'
Finalize 'lgb_nl_3_LR0.05'
Finalize 'lgb_nl_3_LR0.75'
Finalize 'lgb_nl_7_msf256'
Finalize 'lgb_nl_7_msf512'
Finalize 'lgb_nl_7_msf512_cs0.5'
Finalize 'lgb_nl_7_msf512_cs0.75'
Finalize 'lgb_nl_7_msf512_cs0.75_ss0.5'
Finalize 'lgb_nl_7_msf512_cs0.75_ss0.75'
Finalize 'lgb_nl_7_msf512_cs0.75_ss0.9'
Finalize 'lgb_nl_7_msf512_cs0.75_ss0.75_l21'
Finalize 'lgb_nl_7_msf512_cs0.75_ss0.75_l210'
Finalize 'lgb_nl_15_msf512_cs0.75_ss0.75_l21'
Finalize 'lgb_nl_15_msf512_cs0.75_ss0.75_l21_lr0.025'


# CB

In [6]:
for i in [3, 4, 5]:
    e_model.set_node('cb_max_depth_{}'.format(i), grp='cb', processor=cb.CatBoostClassifier,
        edges = {'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
        params={'task_type': 'CPU', 'max_depth': i, 'cat_features': ColSelector(col_type = 'category')},
    )
e_model.exp()

Experimenting 3 node(s)
Exp 1/1 (100%)                          
Experimentation complete: 3 node(s)


In [7]:
e_model.get_collector('AUC').get_metrics_agg('cb_.*')[0].sort_values('valid', ascending = False)

,valid,train_sub,valid_sub
cb_max_depth_5,0.915614,0.919260,0.916666
cb_max_depth_4,0.915582,0.918377,0.916524
cb_max_depth_3,0.915478,0.917804,0.916394


In [8]:
e_model.set_grp('cb_max_depth_5g', parent='cb', processor=cb.CatBoostClassifier,
        edges = {'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
        params={'task_type': 'CPU', 'max_depth': 5, 'cat_features': ColSelector(col_type = 'category')})
e_model.set_node('cb_max_depth_5_lr0.025', grp='cb_max_depth_5g', params = {'learning_rate': 0.025})
e_model.set_node('cb_max_depth_5_lr0.05', grp='cb_max_depth_5g', params = {'learning_rate': 0.05})
e_model.exp()

Experimenting 2 node(s)
Exp 1/1 (100%)                                  
Experimentation complete: 2 node(s)


In [9]:
e_model.get_collector('AUC').get_metrics_agg('cb_.*')[0].sort_values('valid', ascending = False)

,valid,train_sub,valid_sub
cb_max_depth_5,0.915614,0.919260,0.916666
cb_max_depth_5_lr0.05,0.915614,0.919260,0.916666
cb_max_depth_5_lr0.025,0.915590,0.918984,0.916600
cb_max_depth_4,0.915582,0.918377,0.916524
cb_max_depth_3,0.915478,0.917804,0.916394


In [10]:
e_model.set_node('cb_max_depth_5_lr0.05_cs0.75', grp='cb_max_depth_5g', params = {'learning_rate': 0.05, 'colsample_bylevel': 0.75})
e_model.set_node('cb_max_depth_5_lr0.05_cs0.9', grp='cb_max_depth_5g', params = {'learning_rate': 0.05, 'colsample_bylevel': 0.9})
e_model.exp()

Experimenting 2 node(s)
Exp 1/1 (100%)                                       
Experimentation complete: 2 node(s)


In [11]:
e_model.get_collector('AUC').get_metrics_agg('cb_.*')[0].sort_values('valid', ascending = False)

,valid,train_sub,valid_sub
cb_max_depth_5_lr0.05_cs0.9,0.915718,0.919822,0.916673
cb_max_depth_5_lr0.05_cs0.75,0.915660,0.919504,0.916692
cb_max_depth_5,0.915614,0.919260,0.916666
cb_max_depth_5_lr0.05,0.915614,0.919260,0.916666
cb_max_depth_5_lr0.025,0.915590,0.918984,0.916600
cb_max_depth_4,0.915582,0.918377,0.916524
cb_max_depth_3,0.915478,0.917804,0.916394


In [13]:
e_model.finalize('cb_.*')

Finalize 'cb_max_depth_3'
Finalize 'cb_max_depth_4'
Finalize 'cb_max_depth_5'
Finalize 'cb_max_depth_5_lr0.025'
Finalize 'cb_max_depth_5_lr0.05'
Finalize 'cb_max_depth_5_lr0.05_cs0.75'
Finalize 'cb_max_depth_5_lr0.05_cs0.9'


# NN

In [6]:
e_model.set_grp('nn', parent = 'clf', processor = NNClassifier, params = {'metrics': ['auc'], 'early_stopping': 10})

{'result': 'skip',
 'grp': <mllabs._pipeline.PipelineGroup at 0x72cda0970470>,
 'affected_nodes': []}

In [7]:
e_model.set_node(
    'nn_64_32_64', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None)]},
    params={'hidden': {'units': [64, 32, 64]}, 'cat_cols':ColSelector(col_type = 'category')}
)
e_model.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > nn_64_32_64 0/1 (0%)

I0000 00:00:1773464557.885360   19531 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5518 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Exp 1/1 (100%)                                                                                                 
Experimentation complete: 1 node(s)


In [10]:
e_model.set_node(
    'nn_256_128_256', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None)]},
    params={'hidden': {'units': [256, 128, 256]}, 'cat_cols':ColSelector(col_type = 'category')}
)
e_model.exp()

Experimenting 1 node(s)
Exp 1/1 (100%)                                                                                                    
Experimentation complete: 1 node(s)


In [12]:
e_model.set_node(
    'nn_512_256_512', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None)]},
    params={'hidden': {'units': [512, 256, 512]}, 'cat_cols':ColSelector(col_type = 'category')}
)
e_model.exp()

Experimenting 1 node(s)
Exp 1/1 (100%)                                                                                                    
Experimentation complete: 1 node(s)


In [14]:
e_model.set_node(
    'nn_512_256_128_256_512', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None)]},
    params={'hidden': {'units': [512, 256, 128, 256, 512]}, 'cat_cols':ColSelector(col_type = 'category')}
)
e_model.exp()

Experimenting 1 node(s)
Exp 1/1 (100%)                                                                                                            
Experimentation complete: 1 node(s)


In [16]:
e_model.set_node(
    'nn_512_32_512', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None)]},
    params={'hidden': {'units': [512, 32, 512]}, 'cat_cols':ColSelector(col_type = 'category')}
)
e_model.exp()

Experimenting 1 node(s)
Exp 1/1 (100%)                                                                                                   
Experimentation complete: 1 node(s)


In [18]:
e_model.set_node(
    'nn_256_128_64', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None)]},
    params={'hidden': {'units': [256, 128, 64]}, 'cat_cols':ColSelector(col_type = 'category')}
)
e_model.exp()

Experimenting 1 node(s)
Exp 1/1 (100%)                                                                                                   
Experimentation complete: 1 node(s)


In [20]:
e_model.set_node(
    'nn_128_64', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None)]},
    params={'hidden': {'units': [128, 64]}, 'cat_cols':ColSelector(col_type = 'category')}
)
e_model.exp()

Experimenting 1 node(s)
Exp 1/1 (100%)                                                                                               
Experimentation complete: 1 node(s)


In [21]:
e_model.get_collector('AUC').get_metrics_agg('nn_.*')[0].sort_values('valid', ascending = False)

,valid,train_sub,valid_sub
nn_128_64,0.912192,0.914186,0.912598
nn_256_128_256,0.912181,0.914893,0.912734
nn_256_128_64,0.912008,0.914583,0.912503
nn_512_256_512,0.911851,0.914894,0.912280
nn_512_32_512,0.911846,0.914447,0.912432
nn_64_32_64,0.911156,0.912663,0.911517
nn_512_256_128_256_512,0.910886,0.912404,0.911289


# Feature Engineering 

In [55]:
e_model.set_node(
    'expr1', grp='pre', processor = ExprProcessor, edges = {'X': [(None, X_num)]},
    params={'dict_expr': {
        'min_ex_chargs': pl.min_horizontal(pl.col('MonthlyCharges') * pl.col('tenure'), pl.col('TotalCharges')),
        'max_ex_chargs': pl.max_horizontal(pl.col('MonthlyCharges') * pl.col('tenure'), pl.col('TotalCharges')),
        'diff_chargs': pl.col('MonthlyCharges') * pl.col('tenure') - pl.col('TotalCharges'),
    }, 'with_columns': False}
)
e_model.build()

Affected 1 dependent node(s): ['xgb_min_ex_chargs']
Building 1 node(s)
Build 1/1 (100%)                 
Build complete: 1 node(s)


In [56]:
e_model.set_grp(
    'xgb_max_depth3_base', parent='xgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
    params = {'max_depth': 3, 'enable_categorical': True, 'device': 'cuda',
        'verbosity': 0, 'random_state': 1,
        'learning_rate': 0.05, 'colsample_bytree': 0.75, 'subsample': 0.9, 'reg_lambda': 0.01}
)

{'result': 'skip',
 'grp': <mllabs._pipeline.PipelineGroup at 0x72cd523afcb0>,
 'affected_nodes': []}

In [57]:
e_model.set_node('xgb_min_ex_chargs', grp = 'xgb_max_depth3_base', edges={'X': [('expr1', None)]})
e_model.exp()

Experimenting 1 node(s)
Exp 1/1 (100%)                                                                       
Experimentation complete: 1 node(s)


In [59]:
e_model.get_collector('AUC').get_metrics_agg('xgb_min_ex_.*')[0].sort_values('valid', ascending=False)

,valid,train_sub,valid_sub
xgb_min_ex_chargs,0.915708,0.92012,0.916817


In [60]:
e_model.set_grp(
    'xgb_max_depth3_base2', parent='xgb', edges={'X': [(None, X_bin + X_nom + X_bin2 + X_num)]}, 
    params = {'max_depth': 3, 'enable_categorical': True, 'device': 'cuda',
        'verbosity': 0, 'random_state': 1,
        'learning_rate': 0.05, 'colsample_bytree': 0.75, 'subsample': 0.9, 'reg_lambda': 0.01}
)

{'result': 'new',
 'grp': <mllabs._pipeline.PipelineGroup at 0x72cd523ad520>,
 'affected_nodes': []}

In [61]:
e_model.set_node('xgb_base_bin2', grp = 'xgb_max_depth3_base')
e_model.exp()

Experimenting 1 node(s)
Exp 1/1 (100%)                                                                   
Experimentation complete: 1 node(s)


In [66]:
e_model.get_collector('AUC').get_metrics_agg('xgb_base_bin2.*')[0].sort_values('valid', ascending=False)

,valid,train_sub,valid_sub
xgb_base_bin2,0.915938,0.919178,0.916546


In [72]:
from sklearn.preprocessing import PolynomialFeatures
e_model.set_node(
    'pf2', grp='pre', processor = PolynomialFeatures, edges = {'X': [(None, X_num)]},
    params={'degree': 2, 'interaction_only': True}
)
e_model.build()

Building 1 node(s)
Build 1/1 (100%)               
Build complete: 1 node(s)


In [76]:
e_model.set_node('xgb_pf2', grp = 'xgb_max_depth3_base2', edges={'X': [('pf2', None)]})
e_model.set_node('xgb_pf2_expr1', grp = 'xgb_max_depth3_base2', edges={'X': [('pf2', None), ('expr1', None)]})
e_model.exp()

Experimenting 1 node(s)
Exp 1/1 (100%)                                                                   
Experimentation complete: 1 node(s)


In [77]:
e_model.get_collector('AUC').get_metrics_agg('xgb_pf2.*')[0].sort_values('valid', ascending=False)

,valid,train_sub,valid_sub
xgb_pf2,0.915776,0.919992,0.916661
xgb_pf2_expr1,0.915662,0.920040,0.916707


In [106]:
e_model.close_exp()